# RLC Circuit Analysis for Dense Plasma Focus

This interactive tutorial introduces the fundamentals of RLC circuits as they apply to Dense Plasma Focus (DPF) devices.

## Learning Objectives

After completing this notebook, you will be able to:

1. Understand the role of RLC circuits in DPF energy storage and delivery
2. Calculate key circuit parameters (resonant frequency, damping, peak current)
3. Simulate and visualize current waveforms
4. Explore how parameter changes affect DPF performance

## 1. Introduction to DPF Circuits

A Dense Plasma Focus device uses a **capacitor bank** to store electrical energy, which is then rapidly discharged through the plasma chamber. The circuit can be modeled as a series RLC circuit:

```
    ┌──────┐     ┌──────┐     ┌──────┐
    │      │     │      │     │      │
    │  C   │─────│  L   │─────│  R   │
    │      │     │      │     │      │
    └──┬───┘     └──────┘     └──┬───┘
       │                         │
       │      ┌────────┐         │
       └──────│ Plasma │─────────┘
              └────────┘
```

Where:
- **C** = Capacitor bank (energy storage)
- **L** = Circuit inductance (including cables, electrodes)
- **R** = Circuit resistance (cables, connections, plasma)

## 2. Setup and Imports

Let's start by importing the necessary libraries.

In [ ]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# DPF2 imports
try:
    from dpf2.circuit_solver import CircuitSolver, RLCCircuit, run_circuit_simulation
    from dpf2.circuit_config import CircuitConfig
    DPF2_AVAILABLE = True
except ImportError:
    print("Note: dpf2 module not found. Using standalone functions.")
    DPF2_AVAILABLE = False

# Configure matplotlib for better display
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12
plt.style.use('seaborn-v0_8-whitegrid')

print("Setup complete!")

## 3. Basic RLC Circuit Theory

### 3.1 Governing Equation

The current in a series RLC circuit satisfies:

$$L\frac{dI}{dt} + RI + \frac{1}{C}\int I\,dt = V_0$$

Or as a second-order ODE:

$$L\frac{d^2Q}{dt^2} + R\frac{dQ}{dt} + \frac{Q}{C} = 0$$

### 3.2 Key Parameters

- **Resonant frequency**: $\omega_0 = \frac{1}{\sqrt{LC}}$
- **Damping coefficient**: $\alpha = \frac{R}{2L}$
- **Quality factor**: $Q = \frac{1}{R}\sqrt{\frac{L}{C}}$
- **Period**: $T = 2\pi\sqrt{LC}$ (undamped)

In [ ]:
def calculate_circuit_parameters(L, R, C, V0):
    """
    Calculate key RLC circuit parameters.
    
    Parameters
    ----------
    L : float
        Inductance in Henries
    R : float
        Resistance in Ohms
    C : float
        Capacitance in Farads
    V0 : float
        Initial voltage in Volts
        
    Returns
    -------
    dict
        Dictionary of calculated parameters
    """
    omega_0 = 1.0 / np.sqrt(L * C)
    alpha = R / (2 * L)
    Q_factor = (1/R) * np.sqrt(L / C)
    period = 2 * np.pi * np.sqrt(L * C)
    E_stored = 0.5 * C * V0**2
    
    # Determine damping regime
    if alpha < omega_0:
        regime = "Underdamped"
        omega_d = np.sqrt(omega_0**2 - alpha**2)
        # Peak current for underdamped case
        I_peak = (V0 / (L * omega_d)) * np.exp(-alpha * np.arctan(omega_d/alpha) / omega_d)
    elif alpha > omega_0:
        regime = "Overdamped"
        omega_d = 0
        I_peak = V0 / (2 * L * (alpha - np.sqrt(alpha**2 - omega_0**2)))
    else:
        regime = "Critically damped"
        omega_d = 0
        I_peak = V0 / (L * np.e * alpha)
    
    return {
        'omega_0': omega_0,
        'f_0': omega_0 / (2 * np.pi),
        'alpha': alpha,
        'Q_factor': Q_factor,
        'period': period,
        'E_stored': E_stored,
        'regime': regime,
        'omega_d': omega_d if alpha < omega_0 else 0,
        'I_peak_approx': I_peak
    }

# Example: Typical small DPF parameters
L = 50e-9    # 50 nH
R = 10e-3    # 10 mΩ
C = 10e-6    # 10 μF
V0 = 20e3    # 20 kV

params = calculate_circuit_parameters(L, R, C, V0)

print("=" * 50)
print("RLC Circuit Parameters")
print("=" * 50)
print(f"Capacitance:     C  = {C*1e6:.1f} μF")
print(f"Inductance:      L  = {L*1e9:.1f} nH")
print(f"Resistance:      R  = {R*1e3:.1f} mΩ")
print(f"Initial voltage: V₀ = {V0/1e3:.1f} kV")
print("-" * 50)
print(f"Stored energy:   E  = {params['E_stored']/1e3:.2f} kJ")
print(f"Resonant freq:   f₀ = {params['f_0']/1e3:.1f} kHz")
print(f"Period:          T  = {params['period']*1e6:.2f} μs")
print(f"Quality factor:  Q  = {params['Q_factor']:.1f}")
print(f"Damping regime:      {params['regime']}")
print(f"Peak current:    I  ≈ {params['I_peak_approx']/1e3:.0f} kA")
print("=" * 50)

## 4. Analytical Solutions

The RLC circuit has three damping regimes:

### 4.1 Underdamped ($\alpha < \omega_0$)

$$I(t) = \frac{V_0}{L\omega_d} e^{-\alpha t} \sin(\omega_d t)$$

where $\omega_d = \sqrt{\omega_0^2 - \alpha^2}$

### 4.2 Critically Damped ($\alpha = \omega_0$)

$$I(t) = \frac{V_0}{L} t e^{-\alpha t}$$

### 4.3 Overdamped ($\alpha > \omega_0$)

$$I(t) = \frac{V_0}{L(s_1 - s_2)}\left(e^{s_1 t} - e^{s_2 t}\right)$$

where $s_{1,2} = -\alpha \pm \sqrt{\alpha^2 - \omega_0^2}$

In [ ]:
def analytical_current(t, L, R, C, V0):
    """
    Calculate analytical current waveform for RLC circuit.
    
    Parameters
    ----------
    t : array-like
        Time array in seconds
    L, R, C, V0 : float
        Circuit parameters (SI units)
        
    Returns
    -------
    I : ndarray
        Current array in Amperes
    """
    alpha = R / (2 * L)
    omega_0 = 1.0 / np.sqrt(L * C)
    
    if np.isclose(alpha, omega_0, rtol=0.01):
        # Critically damped
        I = (V0 / L) * t * np.exp(-alpha * t)
    elif alpha > omega_0:
        # Overdamped
        s1 = -alpha + np.sqrt(alpha**2 - omega_0**2)
        s2 = -alpha - np.sqrt(alpha**2 - omega_0**2)
        I = (V0 / L) * (np.exp(s1 * t) - np.exp(s2 * t)) / (s1 - s2)
    else:
        # Underdamped (most common for DPF)
        omega_d = np.sqrt(omega_0**2 - alpha**2)
        I = (V0 / (L * omega_d)) * np.exp(-alpha * t) * np.sin(omega_d * t)
    
    return I

def analytical_voltage(t, L, R, C, V0):
    """
    Calculate capacitor voltage for RLC circuit.
    """
    alpha = R / (2 * L)
    omega_0 = 1.0 / np.sqrt(L * C)
    
    if alpha < omega_0:
        omega_d = np.sqrt(omega_0**2 - alpha**2)
        V = V0 * np.exp(-alpha * t) * (np.cos(omega_d * t) + (alpha/omega_d) * np.sin(omega_d * t))
    else:
        # Simplified for non-oscillatory case
        s1 = -alpha + np.sqrt(alpha**2 - omega_0**2 + 1e-20)
        s2 = -alpha - np.sqrt(alpha**2 - omega_0**2 + 1e-20)
        V = V0 * (s1 * np.exp(s2 * t) - s2 * np.exp(s1 * t)) / (s1 - s2)
    
    return V

## 5. Simulating and Visualizing Current Waveforms

Let's simulate a typical DPF discharge and visualize the results.

In [ ]:
# Time array (0 to 10 μs)
t = np.linspace(0, 10e-6, 1000)

# Calculate current and voltage
I = analytical_current(t, L, R, C, V0)
V = analytical_voltage(t, L, R, C, V0)

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Current plot
ax1.plot(t * 1e6, I / 1e3, 'b-', linewidth=2, label='Current')
ax1.set_ylabel('Current (kA)', fontsize=12)
ax1.set_title('DPF Circuit Discharge Waveform', fontsize=14)
ax1.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# Find and mark peak current
I_max = np.max(I)
t_peak = t[np.argmax(I)]
ax1.plot(t_peak * 1e6, I_max / 1e3, 'ro', markersize=10)
ax1.annotate(f'Peak: {I_max/1e3:.0f} kA @ {t_peak*1e6:.2f} μs',
             xy=(t_peak * 1e6, I_max / 1e3),
             xytext=(t_peak * 1e6 + 1, I_max / 1e3 * 0.9),
             fontsize=10,
             arrowprops=dict(arrowstyle='->', color='red'))

# Voltage plot
ax2.plot(t * 1e6, V / 1e3, 'r-', linewidth=2, label='Capacitor Voltage')
ax2.set_xlabel('Time (μs)', fontsize=12)
ax2.set_ylabel('Voltage (kV)', fontsize=12)
ax2.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nPeak current: {I_max/1e3:.1f} kA at t = {t_peak*1e6:.2f} μs")
print(f"Current reversal at: t ≈ {params['period']/2*1e6:.2f} μs")

## 6. Using the dpf2 Circuit Solver

The dpf2 package provides a more sophisticated circuit solver that can handle:
- Time-varying plasma inductance
- Multiple circuit stages
- Crowbar switches
- Coupled plasma dynamics

In [ ]:
if DPF2_AVAILABLE:
    # Create circuit using dpf2
    circuit = RLCCircuit(
        L=50e-9,   # 50 nH
        R=10e-3,   # 10 mΩ
        C=10e-6,   # 10 μF
        V0=20e3    # 20 kV
    )
    
    solver = CircuitSolver(circuit)
    
    # Solve using analytical method
    t_dpf2, I_dpf2 = solver.solve(t_end=10e-6, dt=10e-9, method='analytical')
    
    # Compare with our analytical solution
    plt.figure(figsize=(10, 6))
    plt.plot(t * 1e6, I / 1e3, 'b-', linewidth=2, label='Manual calculation')
    plt.plot(t_dpf2 * 1e6, I_dpf2 / 1e3, 'r--', linewidth=2, label='dpf2 solver')
    plt.xlabel('Time (μs)')
    plt.ylabel('Current (kA)')
    plt.title('Comparison: Manual vs dpf2 Solver')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("dpf2 module not available. Skipping dpf2 solver demonstration.")
    print("To use the full dpf2 functionality, install with: pip install dpf2")

## 7. Parameter Exploration

### 7.1 Effect of Capacitance

Let's explore how changing the capacitance affects the discharge.

In [ ]:
# Vary capacitance
capacitances = [5e-6, 10e-6, 20e-6, 40e-6]  # 5, 10, 20, 40 μF
colors = ['blue', 'green', 'orange', 'red']

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Current waveforms
ax1 = axes[0, 0]
for C_val, color in zip(capacitances, colors):
    I_c = analytical_current(t, L, R, C_val, V0)
    ax1.plot(t * 1e6, I_c / 1e3, color=color, linewidth=2, 
             label=f'C = {C_val*1e6:.0f} μF')

ax1.set_xlabel('Time (μs)')
ax1.set_ylabel('Current (kA)')
ax1.set_title('Effect of Capacitance on Current')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Peak current vs capacitance
ax2 = axes[0, 1]
C_range = np.linspace(1e-6, 50e-6, 50)
I_peaks = [np.max(analytical_current(t, L, R, c, V0)) for c in C_range]
E_stored = 0.5 * C_range * V0**2

ax2.plot(C_range * 1e6, np.array(I_peaks) / 1e3, 'b-', linewidth=2)
ax2.set_xlabel('Capacitance (μF)')
ax2.set_ylabel('Peak Current (kA)', color='blue')
ax2.tick_params(axis='y', labelcolor='blue')
ax2.grid(True, alpha=0.3)

ax2_twin = ax2.twinx()
ax2_twin.plot(C_range * 1e6, E_stored / 1e3, 'r--', linewidth=2)
ax2_twin.set_ylabel('Stored Energy (kJ)', color='red')
ax2_twin.tick_params(axis='y', labelcolor='red')
ax2.set_title('Peak Current and Stored Energy vs Capacitance')

# Period vs capacitance
ax3 = axes[1, 0]
periods = 2 * np.pi * np.sqrt(L * C_range)
ax3.plot(C_range * 1e6, periods * 1e6, 'g-', linewidth=2)
ax3.set_xlabel('Capacitance (μF)')
ax3.set_ylabel('Period (μs)')
ax3.set_title('Discharge Period vs Capacitance')
ax3.grid(True, alpha=0.3)

# Energy in plasma vs time
ax4 = axes[1, 1]
for C_val, color in zip(capacitances, colors):
    I_c = analytical_current(t, L, R, C_val, V0)
    V_c = analytical_voltage(t, L, R, C_val, V0)
    E_cap = 0.5 * C_val * V_c**2
    E_ind = 0.5 * L * I_c**2
    E_total = 0.5 * C_val * V0**2
    E_dissipated = E_total - E_cap - E_ind
    ax4.plot(t * 1e6, E_dissipated / 1e3, color=color, linewidth=2,
             label=f'C = {C_val*1e6:.0f} μF')

ax4.set_xlabel('Time (μs)')
ax4.set_ylabel('Energy Dissipated (kJ)')
ax4.set_title('Energy Delivered to Plasma')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 7.2 Effect of Inductance

Inductance is critical for DPF performance. Lower inductance means faster current rise.

In [ ]:
# Vary inductance
inductances = [20e-9, 50e-9, 100e-9, 200e-9]  # 20, 50, 100, 200 nH

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Current waveforms
ax1 = axes[0]
for L_val, color in zip(inductances, colors):
    I_L = analytical_current(t, L_val, R, C, V0)
    ax1.plot(t * 1e6, I_L / 1e3, color=color, linewidth=2,
             label=f'L = {L_val*1e9:.0f} nH')

ax1.set_xlabel('Time (μs)')
ax1.set_ylabel('Current (kA)')
ax1.set_title('Effect of Inductance on Current Waveform')
ax1.legend()
ax1.grid(True, alpha=0.3)

# dI/dt (current rise rate)
ax2 = axes[1]
for L_val, color in zip(inductances, colors):
    I_L = analytical_current(t, L_val, R, C, V0)
    dIdt = np.gradient(I_L, t)
    ax2.plot(t * 1e6, dIdt / 1e12, color=color, linewidth=2,
             label=f'L = {L_val*1e9:.0f} nH')

ax2.set_xlabel('Time (μs)')
ax2.set_ylabel('dI/dt (kA/μs)')
ax2.set_title('Current Rise Rate')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey insight: Lower inductance → faster current rise → better pinch compression")
print("DPF designers minimize inductance with low-inductance capacitors and short connections.")

## 8. Exercises

### Exercise 1: Calculate Circuit Parameters

For a DPF with:
- Capacitance: 30 μF
- Inductance: 40 nH
- Resistance: 5 mΩ
- Voltage: 25 kV

Calculate:
1. Stored energy
2. Resonant frequency
3. Peak current (approximate)
4. Quarter period (time to peak current)

In [ ]:
# Exercise 1: Your solution here
# ==============================

# Given parameters
C_ex1 = 30e-6    # TODO: Convert to SI units
L_ex1 = 40e-9    # TODO: Convert to SI units
R_ex1 = 5e-3     # TODO: Convert to SI units
V0_ex1 = 25e3    # TODO: Convert to SI units

# Calculate stored energy
E_stored_ex1 = None  # TODO: E = 0.5 * C * V^2

# Calculate resonant frequency
f0_ex1 = None  # TODO: f = 1/(2π√LC)

# Calculate approximate peak current
I_peak_ex1 = None  # TODO: I ≈ V0 / √(L/C) for low damping

# Calculate quarter period
T_quarter_ex1 = None  # TODO: T/4 = π√LC / 2

# Print your results
# print(f"Stored energy: {E_stored_ex1/1e3:.2f} kJ")
# print(f"Resonant frequency: {f0_ex1/1e3:.1f} kHz")
# print(f"Peak current: {I_peak_ex1/1e3:.0f} kA")
# print(f"Quarter period: {T_quarter_ex1*1e6:.2f} μs")

### Exercise 2: Compare Damping Regimes

Create a plot showing underdamped, critically damped, and overdamped circuits.

Use C = 10 μF, L = 100 nH, V0 = 15 kV, and vary R.

In [ ]:
# Exercise 2: Your solution here
# ==============================

C_ex2 = 10e-6
L_ex2 = 100e-9
V0_ex2 = 15e3

# Critical resistance: R_c = 2√(L/C)
R_critical = 2 * np.sqrt(L_ex2 / C_ex2)
print(f"Critical resistance: {R_critical*1e3:.1f} mΩ")

# TODO: Create three resistance values
# R_underdamped = ?  (less than R_critical)
# R_critical = R_critical
# R_overdamped = ?  (greater than R_critical)

# TODO: Calculate current for each case
# TODO: Plot all three on the same graph

# Your plotting code here...

### Exercise 3: Design a DPF Circuit

Design a circuit to achieve:
- Peak current: 200 kA
- Quarter period: 1.5 μs
- Stored energy: 5 kJ

Find the required C, L, and V0.

In [ ]:
# Exercise 3: Your solution here
# ==============================

# Target specifications
I_target = 200e3     # 200 kA
T_quarter = 1.5e-6   # 1.5 μs
E_target = 5e3       # 5 kJ

# Hints:
# 1. From T/4 = π√LC/2, you can get √LC
# 2. For underdamped circuit: I_peak ≈ V0/√(L/C) = V0 * √(C/L)
# 3. E = 0.5 * C * V0^2

# TODO: Solve the system of equations
# sqrt_LC = ?
# From I_peak and E, find C and V0
# Then find L

# Your solution here...

## 9. Summary

In this tutorial, you learned:

1. **RLC circuit basics**: DPF devices use capacitor banks modeled as RLC circuits
2. **Key parameters**: Resonant frequency, damping, Q-factor determine circuit behavior
3. **Analytical solutions**: Different damping regimes produce different waveforms
4. **Design considerations**:
   - Higher C → more energy, slower discharge
   - Lower L → faster current rise, higher peak current
   - Lower R → less damping, more oscillatory

### Key Equations

| Parameter | Formula |
|-----------|--------|
| Stored energy | $E = \frac{1}{2}CV_0^2$ |
| Resonant frequency | $\omega_0 = \frac{1}{\sqrt{LC}}$ |
| Peak current | $I_{peak} \approx \frac{V_0}{\sqrt{L/C}}$ |
| Quarter period | $T/4 = \frac{\pi}{2}\sqrt{LC}$ |
| Critical damping | $R_c = 2\sqrt{L/C}$ |

## 10. Further Reading

- [Magnetic Pressure and Pinch Physics](../plasma_fundamentals/magnetic_pressure.md)
- [Bennett Relation](../plasma_fundamentals/bennett_relation.md)
- [Advanced: Validation Against Experimental Data](../advanced/validation_against_data.ipynb)